# Digit clustering control — offline by default

Version 2. This validates generic medoids, not JDR. The default uses bundled sklearn digits to avoid a network dependency; set USE_MNIST=True for the original MNIST control. Labels are taken from the selected subset.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'jdr.py').exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd


In [ ]:
from sklearn.datasets import load_digits, fetch_openml
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from src.JDRKMedoids import JDRKMedoids
from src.evaluation import evaluate_clustering
USE_MNIST = False
if USE_MNIST:
    X,y = fetch_openml('mnist_784', version=1, return_X_y=True, as_frame=False)
else:
    X,y = load_digits(return_X_y=True)
rng = np.random.default_rng(42)
indices = rng.choice(len(X), 100, replace=False)
X_sub,y_sub = X[indices],y[indices].astype(int)
Z = PCA(n_components=20, random_state=42).fit_transform(X_sub)
D = pairwise_distances(Z)
np.fill_diagonal(D,0)
models = [JDRKMedoids(10, random_state=42+i).fit(D) for i in range(5)]
model = min(models,key=lambda m:m.inertia_)
print('Medoid true labels:', y_sub[model.medoids_])
evaluate_clustering(y_sub,model.labels_)
